In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("run_id", "manual")
run_id = dbutils.widgets.get("run_id")

In [0]:
df_silver = (
    spark.read
        .table("databricksbootcamp.greeengridcasestudy.silver_smartbill")
        .drop("ingested_at", "run_id")
)

df_gold = (
    df_silver
    .withColumn(
        "period",
        F.when(
            (F.col("Hour") >= 18) & (F.col("Hour") <= 21),
            F.lit("peak")
        ).otherwise(F.lit("offpeak"))
    )
    .groupBy("meter_id")
    .pivot("period", ["peak", "offpeak"])
    .agg(
        F.round(F.sum("kwh"), 2).alias("kwh"),
        F.round(F.sum("Bill_Amount"), 2).alias("bill")
    )
    .withColumn("peak_kwh", F.coalesce(F.col("peak_kwh"), F.lit(0.0)))
    .withColumn("offpeak_kwh", F.coalesce(F.col("offpeak_kwh"), F.lit(0.0)))
    .withColumn("peak_bill", F.coalesce(F.col("peak_bill"), F.lit(0.0)))
    .withColumn("offpeak_bill", F.coalesce(F.col("offpeak_bill"), F.lit(0.0)))
    .withColumn(
        "total_kwh",
        F.round(F.col("peak_kwh") + F.col("offpeak_kwh"), 2)
    )
    .withColumn(
        "total_bill",
        F.round(F.col("peak_bill") + F.col("offpeak_bill"), 2)
    )
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("run_id", F.lit(str(run_id)))
)

In [0]:
df_gold.write.mode("overwrite").saveAsTable("databricksbootcamp.greeengridcasestudy.gold_daily_usage")